<a href="https://colab.research.google.com/github/TamirPalay/DI_Exercises/blob/main/week14/day3-4/Pinecone_Rerankin_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 12.7 MB/s eta 0:00:00


In [4]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [5]:
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

## Define query and documents

In [6]:
query = "Tell me about Apple's products"
documents = [
    "Apples are a fructose rich fruit.", # Add a document about apple fruit
    "Apple is known for creating innovative smartphone solutions. They are most famous for the iphone and mac computer", # Add a document about Apple company products
    "They say that eating an apple a day keeps the doctor away.", # Add another fruit-related document
    "Apple is one of the most reliable stock investment choices of the decade as the iphone is still one of the most requested products on the market.", # Add another company-related document
    "At 20 weeks pregnant, the foetus is roughly the size of an apple." # Add one more document (your choice)
]

## Calling reranker

In [7]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3 # e.g., 3
)

In [8]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    if matches and matches.data:
        for i, m in enumerate(matches.data):
            print("query ", i+1) # Print the position (i+1), m.score, and m.document.text
            print("score: ", m.score)
            print("text: ", m.document.text,"/n")
    else:
        print("No reranked results found or an issue occurred during reranking.")

show_reranked_results(query, reranked)

Query: Tell me about Apple's products
query  1
score:  0.6016521
text:  Apple is known for creating innovative smartphone solutions. They are most famous for the iphone and mac computer /n
query  2
score:  0.25720495
text:  Apple is one of the most reliable stock investment choices of the decade as the iphone is still one of the most requested products on the market. /n
query  3
score:  0.025035424
text:  Apples are a fructose rich fruit. /n


# Part 2: Serverless Index for Medical notes

In [9]:
!pip install pandas torch transformers

In [10]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'med-notes-index' # Give your index a name

In [11]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384, # This matches our embedding model size
    metric='cosine', # Distance metric for similarity
    spec=spec
)

{
    "name": "med-notes-index",
    "metric": "cosine",
    "host": "med-notes-index-7v6jlnd.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

## Part 3: Loading sample data

In [12]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/refs/heads/main/Week%208/W8D2/data.json" # Insert the GitHub raw URL here
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

In [15]:
# Show head of the DataFrame
print("Data shape:", df.shape) # Show number of rows and columns
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


## Upserting Data into the index

In [16]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df) # Pass the DataFrame

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

In [18]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)

    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

Vector count:  100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

## Query and Embedding the function

In [19]:
def get_embedding(input_question):
  model_name = 'sentence-transformers/all-MiniLM-L6-v2'
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name)
  encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
  with torch.no_grad():
    model_output = model(**encoded_input)
    # Which dimension to average?
  embedding = model_output.last_hidden_state[0].mean(dim=0)
  return embedding

In [20]:
# Build a query to search
question = "what should I do if I have a headache and thirst at the same time?" # Ask a medical question
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Display and Rerank clinical notes

In [22]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}') # What field contains the score?
        print(f' Metadata: {match["metadata"]}') # What field contains metadata?
        print('')

show_results(question, sorted_matches)

Question: 'what should I do if I have a headache and thirst at the same time?'

Results:
   1. ID: P011
 Score: 0.557904065
 Metadata: {'advice': 'rest, hydrate', 'symptoms': 'headache'}

   2. ID: P071
 Score: 0.412137806
 Metadata: {'advice': 'headache diary', 'symptoms': 'headaches'}

   3. ID: P044
 Score: 0.335938871
 Metadata: {'condition': 'dehydration', 'treatment': 'IV fluids'}

   4. ID: P092
 Score: 0.335938871
 Metadata: {'condition': 'dehydration', 'treatment': 'IV fluids'}

   5. ID: P040
 Score: 0.301095307
 Metadata: {'condition': 'chronic migraines', 'referral': 'neurology'}



In [26]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

In [32]:
# Define a more specific query for reranking
refined_query = "What should I do for a dehydrated patient?" # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3, # How many top results do you want?
    return_documents=True,
)

In [33]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    if matches and matches.data:
        for i, match in enumerate(matches.data):
            print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
            print(f' Score: {match.score}') # What attribute contains the reranking score?
            print(f' Reranking Field: {match.document.reranking_field}') # What contains the searchable field?
            print('')
    else:
        print("No reranked results found or an issue occurred during reranking.")
show_reranked_results(refined_query, reranked_results)

Question: 'What should I do for a dehydrated patient?'

Reranked Results:
   1. ID: P092
 Score: 0.025957357
 Reranking Field: condition: dehydration; treatment: IV fluids

   2. ID: P044
 Score: 0.025711587
 Reranking Field: condition: dehydration; treatment: IV fluids

   3. ID: P011
 Score: 0.0008559007
 Reranking Field: advice: rest, hydrate; symptoms: headache

